# Fine-tuning de modelos transformer para clasificación de candidatos terminológicos

Este cuaderno implementa el fine-tuning supervisado de modelos Transformer
(BERT, FinBERT, RoBERTa) para la clasificación binaria de candidatos
como términos financieros relevantes o no relevantes.

⚠️ Nota: el entrenamiento se realizó en Google Colab con GPU.
Los pesos entrenados no se incluyen en este repositorio.

In [ ]:
!git clone https://github.com/AntoJimA/tfg-terminologia-financiera
%cd tfg-terminologia-financiera
!git checkout finetuning

Cloning into 'tfg-terminologia-financiera'...
remote: Enumerating objects: 2244, done.
remote: Counting objects: 100% (2244/2244), done.
remote: Compressing objects: 100% (1367/1367), done.
remote: Total 2244 (delta 47), reused 2235 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (2244/2244), 4.81 MiB | 12.48 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/tfg-terminologia-financiera
Branch 'finetuning' set up to track remote branch 'finetuning' from 'origin'.
Switched to a new branch 'finetuning'


In [ ]:
!python scripts/build_candidate_dataset_for_finetuning.py

In [ ]:
!ls data
!ls scripts

dev.jsonl  test.jsonl  train.jsonl
build_candidate_dataset_for_finetuning.py
eval_keywords.py
inspect_econstor_with_predictions.py
prepare_econstor_for_mderank.py
prepare_econstor_for_rankers.py
rebuild_econstor_docs_for_attentionrank.py


In [ ]:
%cd /content
!ls
%cd /content/tfg-terminologia-financiera
!pwd
!ls

/content
sample_data  tfg-terminologia-financiera
/content/tfg-terminologia-financiera
/content/tfg-terminologia-financiera
 attentionrank		  run_attentionrank_bert_full.sh
 data			  run_attentionrank_finbert_full.sh
 econstor		  run_attentionrank_flangbert_full.sh
 harvest_econstor.py	  run_attentionrank_flang_full.sh
 mdeRank		  run_attentionrank_step5_and_export.sh
 notas-attentionRank.md   run_mderank_bert_econstor.sh
 notas-mderank.md	  run_mderank_bert.sh
 nuevoarchivo.sql	  run_mderank_finbert_econstor.sh
 ODECO.json		  run_mderank_finbert.sh
'primer experimento.md'   scripts
 README.md		  split-econstor.py
 results


In [ ]:
!python scripts/build_candidate_dataset_for_finetuning.py


In [ ]:
%%writefile scripts/train_finetune_finbert.py
#!/usr/bin/env python
"""
Fine-tuning de FinBERT para clasificar candidatos como keyword (1) o no (0).

Lee:
  - data/train_pairs.jsonl
  - data/dev_pairs.jsonl
  - data/test_pairs.jsonl

Cada línea:
  {
    "doc_id": ...,
    "text": "...",
    "candidate": "...",
    "label": 0/1
  }
"""

from pathlib import Path
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "models" / "finbert_finetuned_candidates"


def load_pair_datasets():
    data_files = {
        "train": str(DATA_DIR / "train_pairs.jsonl"),
        "validation": str(DATA_DIR / "dev_pairs.jsonl"),
        "test": str(DATA_DIR / "test_pairs.jsonl"),
    }
    ds = load_dataset("json", data_files=data_files)
    return ds


def build_tokenizer_and_model():
    model_name = "ProsusAI/finbert"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
    )
    return tokenizer, model


def preprocess_function_factory(tokenizer, max_length: int = 256):
    sep_token = tokenizer.sep_token or "[SEP]"

    def preprocess(examples):
        texts = examples["text"]
        cands = examples["candidate"]
        inputs = [f"{cand} {sep_token} {txt}" for cand, txt in zip(cands, texts)]
        enc = tokenizer(
            inputs,
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        enc["labels"] = examples["label"]
        return enc

    return preprocess


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def main():
    print("📂 Cargando datasets supervisados (pairs)...")
    raw_datasets = load_pair_datasets()
    print(raw_datasets)

    print("🧠 Cargando tokenizer + modelo (FinBERT)...")
    tokenizer, model = build_tokenizer_and_model()

    print("🔧 Tokenizando ejemplos...")
    preprocess_fn = preprocess_function_factory(tokenizer, max_length=256)

    tokenized_datasets = raw_datasets.map(
        preprocess_fn,
        batched=True,
        remove_columns=["doc_id", "text", "candidate"],
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=100,
        save_total_limit=2,
    )

    print("🚀 Iniciando entrenamiento (FinBERT fine-tuning)...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    print("📏 Evaluando sobre test_pairs.jsonl...")
    test_metrics = trainer.evaluate(tokenized_datasets["test"])
    print("✅ Métricas en test:", test_metrics)

    print(f"💾 Guardando modelo en: {OUTPUT_DIR}")
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))


if __name__ == "__main__":
    main()

Writing scripts/train_finetune_finbert.py


In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!pip install transformers datasets accelerate evaluate scikit-learn
!pip install spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!pwd
!ls

In [ ]:
%cd /content/drive/MyDrive
!mkdir -p tfg
%cd tfg
!git clone https://github.com/AntoJimA/tfg-terminologia-financiera.git
%cd tfg-terminologia-financiera
!ls

/content/drive/MyDrive
/content/drive/MyDrive/tfg
Cloning into 'tfg-terminologia-financiera'...
remote: Enumerating objects: 2244, done.
remote: Counting objects: 100% (2244/2244), done.
remote: Compressing objects: 100% (1367/1367), done.
remote: Total 2244 (delta 47), reused 2235 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (2244/2244), 4.81 MiB | 4.14 MiB/s, done.
Resolving deltas: 100% (47/47), done.
Updating files: 100% (1665/1665), done.
/content/drive/MyDrive/tfg/tfg-terminologia-financiera
attentionrank		ODECO.json
data			README.md
econstor		results
harvest_econstor.py	run_attentionrank_bert_full.sh
mdeRank			run_attentionrank_step5_and_export.sh
notas-attentionRank.md	scripts
notas-mderank.md	split-econstor.py
nuevoarchivo.sql


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
%cd /content/drive/MyDrive/tfg/tfg-terminologia-financiera
!ls

/content/drive/MyDrive/tfg/tfg-terminologia-financiera
attentionrank		ODECO.json
data			README.md
econstor		results
harvest_econstor.py	run_attentionrank_bert_full.sh
mdeRank			run_attentionrank_step5_and_export.sh
models			scripts
notas-attentionRank.md	split-econstor.py
notas-mderank.md	wandb
nuevoarchivo.sql


In [3]:
!pip install -q transformers datasets accelerate scikit-learn spacy sentencepiece
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 138.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!git chekout finetuning

git: 'chekout' is not a git command. See 'git --help'.

The most similar command is
	checkout


In [ ]:
%cd /content/drive/MyDrive/tfg/tfg-terminologia-financiera
!ls

/content/drive/MyDrive/tfg/tfg-terminologia-financiera
attentionrank		nuevoarchivo.sql
data			ODECO.json
econstor		README.md
harvest_econstor.py	results
mdeRank			run_attentionrank_bert_full.sh
models			run_attentionrank_step5_and_export.sh
notas-attentionRank.md	scripts
notas-mderank.md	split-econstor.py


In [ ]:
%%writefile scripts/build_candidate_dataset_for_finetuning.py
from pathlib import Path
import json
import random
from typing import List, Dict
import spacy
from tqdm import tqdm

ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"

TRAIN_IN = DATA_DIR / "train.jsonl"
DEV_IN   = DATA_DIR / "dev.jsonl"
TEST_IN  = DATA_DIR / "test.jsonl"

TRAIN_OUT = DATA_DIR / "train_pairs.jsonl"
DEV_OUT   = DATA_DIR / "dev_pairs.jsonl"
TEST_OUT  = DATA_DIR / "test_pairs.jsonl"

# --------------------------------------------------------------------
# Utilidades básicas
# --------------------------------------------------------------------

def load_jsonl(path: Path) -> List[Dict]:
    examples = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            examples.append(json.loads(line))
    return examples

def save_jsonl(path: Path, data: List[Dict]):
    with path.open("w", encoding="utf-8") as f:
        for obj in data:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def normalize(text: str) -> str:
    return " ".join(text.lower().strip().split())

# --------------------------------------------------------------------
# Construcción de pares (texto, candidato, label)
# label = 1 si el candidato es una keyword gold, 0 si es negativo
# --------------------------------------------------------------------

def build_pairs(examples: List[Dict], nlp, max_neg_per_doc: int = 20) -> List[Dict]:
    pairs = []
    for ex in tqdm(examples, desc="Construyendo pairs"):
        doc_id = ex.get("doc_id", None)
        text   = ex["text"]
        gold   = ex["keywords"]  # asumimos lista de strings

        text_norm = normalize(text)
        gold_norm = [normalize(k) for k in gold]

        # 1) Candidatos por noun chunks (spaCy)
        doc_spacy = nlp(text)
        cand_set = set()
        for chunk in doc_spacy.noun_chunks:
            cand = normalize(chunk.text)
            if len(cand) < 3:
                continue
            cand_set.add(cand)

        # 2) Positivos: keywords gold que aparezcan en el texto
        used_pos = set()
        for kw_norm, kw_original in zip(gold_norm, gold):
            if kw_norm and kw_norm in text_norm:
                if kw_norm in used_pos:
                    continue
                used_pos.add(kw_norm)
                pairs.append({
                    "doc_id": doc_id,
                    "text": text,
                    "candidate": kw_original,
                    "label": 1,
                })

        # 3) Negativos: candidatos de spaCy que no son gold
        neg_candidates = [c for c in cand_set if c not in gold_norm]
        random.shuffle(neg_candidates)
        neg_candidates = neg_candidates[:max_neg_per_doc]

        for cand in neg_candidates:
            pairs.append({
                "doc_id": doc_id,
                "text": text,
                "candidate": cand,
                "label": 0,
            })

    return pairs

def main():
    print(f"📂 Leyendo ficheros de entrada desde {DATA_DIR}")
    train_ex = load_jsonl(TRAIN_IN)
    dev_ex   = load_jsonl(DEV_IN)
    test_ex  = load_jsonl(TEST_IN)

    print(f"  Train: {len(train_ex)} docs")
    print(f"  Dev:   {len(dev_ex)} docs")
    print(f"  Test:  {len(test_ex)} docs")

    print("🔁 Cargando spaCy (en_core_web_sm)...")
    import spacy
    nlp = spacy.load("en_core_web_sm")

    print("🧱 Construyendo pairs para train...")
    train_pairs = build_pairs(train_ex, nlp, max_neg_per_doc=20)

    print("🧱 Construyendo pairs para dev...")
    dev_pairs   = build_pairs(dev_ex, nlp, max_neg_per_doc=20)

    print("🧱 Construyendo pairs para test...")
    test_pairs  = build_pairs(test_ex, nlp, max_neg_per_doc=20)

    print("💾 Guardando:")
    print(f"  {TRAIN_OUT}  ({len(train_pairs)} pairs)")
    print(f"  {DEV_OUT}    ({len(dev_pairs)} pairs)")
    print(f"  {TEST_OUT}   ({len(test_pairs)} pairs)")

    save_jsonl(TRAIN_OUT, train_pairs)
    save_jsonl(DEV_OUT,   dev_pairs)
    save_jsonl(TEST_OUT,  test_pairs)

    print("✅ Dataset de fine-tuning generado.")

if __name__ == "__main__":
    main()

Overwriting scripts/build_candidate_dataset_for_finetuning.py


In [ ]:
!python scripts/build_candidate_dataset_for_finetuning.py

📂 Leyendo ficheros de entrada desde /content/drive/MyDrive/tfg/tfg-terminologia-financiera/data
  Train: 6400 docs
  Dev:   800 docs
  Test:  800 docs
🔁 Cargando spaCy (en_core_web_sm)...
🧱 Construyendo pairs para train...
Construyendo pairs: 100% 6400/6400 [03:07<00:00, 34.12it/s]
🧱 Construyendo pairs para dev...
Construyendo pairs: 100% 800/800 [00:21<00:00, 36.53it/s]
🧱 Construyendo pairs para test...
Construyendo pairs: 100% 800/800 [00:23<00:00, 34.77it/s]
💾 Guardando:
  /content/drive/MyDrive/tfg/tfg-terminologia-financiera/data/train_pairs.jsonl  (144203 pairs)
  /content/drive/MyDrive/tfg/tfg-terminologia-financiera/data/dev_pairs.jsonl    (18008 pairs)
  /content/drive/MyDrive/tfg/tfg-terminologia-financiera/data/test_pairs.jsonl   (18100 pairs)
✅ Dataset de fine-tuning generado.


In [ ]:
!ls data

dev.jsonl	 test.jsonl	   train.jsonl
dev_pairs.jsonl  test_pairs.jsonl  train_pairs.jsonl


In [ ]:
%%writefile scripts/train_finetune_finbert.py
#!/usr/bin/env python
"""
Fine-tuning de FinBERT para clasificar candidatos como keyword (1) o no (0).

Usa:
  - data/train_pairs.jsonl
  - data/dev_pairs.jsonl
  - data/test_pairs.jsonl

Cada línea:
  {
    "doc_id": ...,
    "text": "...",
    "candidate": "...",
    "label": 0/1
  }
"""

from pathlib import Path
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score


# Rutas base
ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "models" / "finbert_finetuned_candidates"


def load_pair_datasets():
    """
    Carga los datasets supervisados generados por build_candidate_dataset_for_finetuning.py:
      - data/train_pairs.jsonl
      - data/dev_pairs.jsonl
      - data/test_pairs.jsonl
    """
    data_files = {
        "train": str(DATA_DIR / "train_pairs.jsonl"),
        "validation": str(DATA_DIR / "dev_pairs.jsonl"),
        "test": str(DATA_DIR / "test_pairs.jsonl"),
    }
    ds = load_dataset("json", data_files=data_files)
    return ds


def build_tokenizer_and_model():
    """
    Carga el tokenizer y el modelo base FinBERT (ProsusAI/finbert),
    pero reemplaza la cabeza de clasificación por una de 2 clases.
    """
    model_name = "ProsusAI/finbert"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,                   # binario: keyword / no keyword
        ignore_mismatched_sizes=True,   # rehace la cabeza si la del checkpoint es de 3 clases
    )
    return tokenizer, model


def preprocess_function_factory(tokenizer, max_length: int = 256):
    """
    Crea una función de preprocesado que concatena:
      [candidate] [SEP] [text]
    y tokeniza con padding a max_length.
    """
    sep_token = tokenizer.sep_token or "[SEP]"

    def preprocess(examples):
        texts = examples["text"]
        cands = examples["candidate"]
        inputs = [f"{cand} {sep_token} {txt}" for cand, txt in zip(cands, texts)]
        enc = tokenizer(
            inputs,
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        enc["labels"] = examples["label"]
        return enc

    return preprocess


def compute_metrics(eval_pred):
    """
    Métricas estándar para clasificación binaria:
      - accuracy
      - precision
      - recall
      - f1
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def main():
    print("📂 Cargando datasets supervisados (pairs)...")
    raw_datasets = load_pair_datasets()
    print(raw_datasets)

# ================================================================
# 🔽 SUBMUESTREO PARA ACELERAR EL ENTRENAMIENTO
# ================================================================
    from datasets import DatasetDict

    MAX_TRAIN = 40000   # ▶️ Puedes poner 20000 si quieres aún más velocidad

    print(f"📉 Submuestreando el dataset train a {MAX_TRAIN} ejemplos...")

    train_subset = raw_datasets["train"].shuffle(seed=42).select(
    range(min(MAX_TRAIN, len(raw_datasets["train"])))
    )

    raw_datasets = DatasetDict({
        "train": train_subset,
        "validation": raw_datasets["validation"],
        "test": raw_datasets["test"],
    })

    print(f"📉 Train reducido a: {len(raw_datasets['train'])} ejemplos")
    print("================================================")
    # ================================================================

    print("🧠 Cargando tokenizer + modelo (FinBERT)...")
    tokenizer, model = build_tokenizer_and_model()

    print("🔧 Tokenizando ejemplos...")
    preprocess_fn = preprocess_function_factory(tokenizer, max_length=256)

    tokenized_datasets = raw_datasets.map(
        preprocess_fn,
        batched=True,
        remove_columns=["doc_id", "text", "candidate"],
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ⚠️ Versión SIMPLE de TrainingArguments (sin evaluation_strategy ni cosas nuevas)
    training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,         # ahora sí, epochs completas
    weight_decay=0.01,
    logging_steps=200,
)

    print("🚀 Iniciando entrenamiento (FinBERT fine-tuning)...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    print("📏 Evaluando sobre test_pairs.jsonl...")
    test_metrics = trainer.evaluate(tokenized_datasets["test"])
    print("✅ Métricas en test:", test_metrics)

    print(f"💾 Guardando modelo en: {OUTPUT_DIR}")
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))


if __name__ == "__main__":
    main()

Overwriting scripts/train_finetune_finbert.py


In [ ]:
!pip install transformers==4.44.2

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

True
Tesla T4


In [ ]:
%cd /content/drive/MyDrive/tfg/tfg-terminologia-financiera
!WANDB_DISABLED=true python scripts/train_finetune_finbert.py

/content/drive/MyDrive/tfg/tfg-terminologia-financiera
2025-12-07 13:57:59.660299: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765115879.694948   16333 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765115879.704826   16333 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765115879.731628   16333 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765115879.731666   16333 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765115879.731674   

In [ ]:
from datasets import load_dataset
from collections import Counter

ds_pairs = load_dataset(
    "json",
    data_files={
        "train": "data/train_pairs.jsonl",
        "validation": "data/dev_pairs.jsonl",
        "test": "data/test_pairs.jsonl",
    }
)

for split in ["train", "validation", "test"]:
    labels = ds_pairs[split]["label"]
    c = Counter(labels)
    total = len(labels)
    print(f"\n{split.upper()}:")
    print(c)
    print({k: f"{v/total:.3f}" for k, v in c.items()})


TRAIN:
Counter({0: 126856, 1: 17347})
{1: '0.120', 0: '0.880'}

VALIDATION:
Counter({0: 15844, 1: 2164})
{1: '0.120', 0: '0.880'}

TEST:
Counter({0: 15872, 1: 2228})
{1: '0.123', 0: '0.877'}


In [ ]:
%%writefile scripts/train_finetune_bert.py
#!/usr/bin/env python
"""
Fine-tuning de BERT base (bert-base-uncased) para clasificar candidatos
como keyword (1) o no (0).

Usa:
  - data/train_pairs.jsonl
  - data/dev_pairs.jsonl
  - data/test_pairs.jsonl

Cada línea:
  {
    "doc_id": ...,
    "text": "...",
    "candidate": "...",
    "label": 0/1
  }
"""

from pathlib import Path
import numpy as np

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score


# Rutas base
ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "models" / "bert_finetuned_candidates"


def load_pair_datasets():
    """
    Carga los datasets supervisados generados por build_candidate_dataset_for_finetuning.py:
      - data/train_pairs.jsonl
      - data/dev_pairs.jsonl
      - data/test_pairs.jsonl
    """
    data_files = {
        "train": str(DATA_DIR / "train_pairs.jsonl"),
        "validation": str(DATA_DIR / "dev_pairs.jsonl"),
        "test": str(DATA_DIR / "test_pairs.jsonl"),
    }
    ds = load_dataset("json", data_files=data_files)
    return ds


def build_tokenizer_and_model():
    """
    Carga el tokenizer y el modelo base BERT (bert-base-uncased),
    con una cabeza de clasificación binaria (2 clases).
    """
    model_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,                 # binario: keyword / no keyword
    )
    return tokenizer, model


def preprocess_function_factory(tokenizer, max_length: int = 256):
    """
    Crea una función de preprocesado que concatena:
      [candidate] [SEP] [text]
    y tokeniza con padding a max_length.
    """
    sep_token = tokenizer.sep_token or "[SEP]"

    def preprocess(examples):
        texts = examples["text"]
        cands = examples["candidate"]
        inputs = [f"{cand} {sep_token} {txt}" for cand, txt in zip(cands, texts)]
        enc = tokenizer(
            inputs,
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        enc["labels"] = examples["label"]
        return enc

    return preprocess


def compute_metrics(eval_pred):
    """
    Métricas estándar para clasificación binaria:
      - accuracy
      - precision
      - recall
      - f1
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def main():
    print("📂 Cargando datasets supervisados (pairs)...")
    raw_datasets = load_pair_datasets()
    print(raw_datasets)

    # ================================================================
    # 🔽 SUBMUESTREO PARA ACELERAR EL ENTRENAMIENTO
    # ================================================================
    MAX_TRAIN = 40000   # ▶️ Pon 20000 si quieres que vaya aún más rápido

    print(f"📉 Submuestreando el dataset train a {MAX_TRAIN} ejemplos...")

    train_subset = raw_datasets["train"].shuffle(seed=42).select(
        range(min(MAX_TRAIN, len(raw_datasets["train"])))
    )

    raw_datasets = DatasetDict({
        "train": train_subset,
        "validation": raw_datasets["validation"],
        "test": raw_datasets["test"],
    })

    print(f"📉 Train reducido a: {len(raw_datasets['train'])} ejemplos")
    print("================================================")

    print("🧠 Cargando tokenizer + modelo (BERT base)...")
    tokenizer, model = build_tokenizer_and_model()

    print("🔧 Tokenizando ejemplos...")
    preprocess_fn = preprocess_function_factory(tokenizer, max_length=256)

    tokenized_datasets = raw_datasets.map(
        preprocess_fn,
        batched=True,
        remove_columns=["doc_id", "text", "candidate"],
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=200,
        report_to="none",   # 🔇 sin wandb ni nada raro
    )

    print("🚀 Iniciando entrenamiento (BERT base fine-tuning)...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    print("📏 Evaluando sobre test_pairs.jsonl...")
    test_metrics = trainer.evaluate(tokenized_datasets["test"])
    print("✅ Métricas en test:", test_metrics)

    print(f"💾 Guardando modelo en: {OUTPUT_DIR}")
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))


if __name__ == "__main__":
    main()

Writing scripts/train_finetune_bert.py


In [4]:
!python scripts/train_finetune_bert.py

2025-12-08 09:48:35.068327: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-08 09:48:35.085863: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765187315.108601    1319 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765187315.115330    1319 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765187315.132742    1319 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [8]:
#!/usr/bin/env python
"""
Fine-tuning de RoBERTa para clasificar candidatos como keyword (1) o no (0).
Usa:
  - data/train_pairs.jsonl
  - data/dev_pairs.jsonl
  - data/test_pairs.jsonl
"""

from pathlib import Path
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score


ROOT = Path(".").resolve()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "models" / "roberta_finetuned_candidates"


def load_pair_datasets():
    data_files = {
        "train": str(DATA_DIR / "train_pairs.jsonl"),
        "validation": str(DATA_DIR / "dev_pairs.jsonl"),
        "test": str(DATA_DIR / "test_pairs.jsonl"),
    }
    return load_dataset("json", data_files=data_files)


def build_tokenizer_and_model():
    model_name = "roberta-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    return tokenizer, model


def preprocess_function_factory(tokenizer, max_length: int = 256):
    sep = tokenizer.sep_token or "</s>"

    def preprocess(examples):
        texts = examples["text"]
        cands = examples["candidate"]
        # RoBERTa no usa token_type_ids -> perfecto
        inputs = [f"{cand} {sep} {txt}" for cand, txt in zip(cands, texts)]
        enc = tokenizer(
            inputs,
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        enc["labels"] = examples["label"]
        return enc

    return preprocess


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


def main():
    print("📂 Cargando datasets supervisados (pairs)...")
    raw = load_pair_datasets()
    print(raw)

    print("🧠 Cargando tokenizer + modelo (RoBERTa)...")
    tokenizer, model = build_tokenizer_and_model()

    print("🔧 Tokenizando ejemplos...")
    tokenized = raw.map(
        preprocess_function_factory(tokenizer, max_length=256),
        batched=True,
        remove_columns=["doc_id", "text", "candidate"],
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # IMPORTANTÍSIMO: desactivar W&B y evitar guardar millones de checkpoints
    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=200,
        report_to="none",          # <--- evita el lío de wandb
        save_strategy="epoch",
        save_total_limit=2,
    )

    print("🚀 Iniciando entrenamiento (RoBERTa fine-tuning)...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    print("📏 Evaluando sobre test_pairs.jsonl...")
    metrics = trainer.evaluate(tokenized["test"])
    print("✅ Métricas en test:", metrics)

    print(f"💾 Guardando modelo en: {OUTPUT_DIR}")
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))


if __name__ == "__main__":
    main()

📂 Cargando datasets supervisados (pairs)...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'text', 'candidate', 'label'],
        num_rows: 144203
    })
    validation: Dataset({
        features: ['doc_id', 'text', 'candidate', 'label'],
        num_rows: 18008
    })
    test: Dataset({
        features: ['doc_id', 'text', 'candidate', 'label'],
        num_rows: 18100
    })
})
🧠 Cargando tokenizer + modelo (RoBERTa)...


...

🚀 Iniciando entrenamiento (RoBERTa fine-tuning)...


/tmp/ipython-input-2320834681.py:106: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
200,0.336800
400,0.263100
600,0.224000
800,0.184900
1000,0.217500
1200,0.187500
1400,0.180100
1600,0.185400
1800,0.182300
2000,0.190200


📏 Evaluando sobre test_pairs.jsonl...


✅ Métricas en test: {'eval_loss': 0.18492218852043152, 'eval_accuracy': 0.9509944751381215, 'eval_precision': 0.8209669698420297, 'eval_recall': 0.7697486535008977, 'eval_f1': 0.7945332406763956, 'eval_runtime': 62.8751, 'eval_samples_per_second': 287.872, 'eval_steps_per_second': 18.004, 'epoch': 3.0}
💾 Guardando modelo en: /content/drive/MyDrive/tfg/tfg-terminologia-financiera/models/roberta_finetuned_candidates


In [9]:
!date

Thu Dec 18 06:33:05 PM UTC 2025
